# Frontend Code Converter

Convert React components to Angular/Svelte/Vue/Vanilla JS, with optional state manager integration


In [ ]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from IPython.display import Markdown, display


In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")



In [ ]:
# Connect to client libraries

openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
grok_url = "https://api.x.ai/v1"
groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)



In [ ]:
models = ["gpt-5", "claude-sonnet-4-5-20250929", "grok-4", "gemini-2.5-pro", "qwen2.5-coder", "kimi-k2.5:cloud", "deepseek-coder-v2", "openai/gpt-oss-20b", "qwen/qwen3-coder-30b-a3b-instruct", "openai/gpt-oss-120b", ]


clients = {
    "gpt-5": openai, 
    "claude-sonnet-4-5-20250929": anthropic, 
    "kimi-k2.5:cloud": ollama,
    "grok-4": openrouter, 
    "gemini-2.5-pro": gemini, 
    "openai/gpt-oss-120b": groq, 
    "qwen2.5-coder": groq, 
    "deepseek-coder-v2": ollama, 
    "openai/gpt-oss-20b": groq, 
    "qwen/qwen3-coder-30b-a3b-instruct": openrouter}

# Want to keep costs ultra-low? Replace this with models of your choice, using the examples from yesterday

In [ ]:
frameworks = ["Angular", "Svelte", "Vue", "Vanilla JS"]

file_extensions = {
    "Angular": "ts",
    "Svelte": "svelte",
    "Vue": "vue",
    "Vanilla JS": "js",
}

code_languages = {
    "Angular": "TypeScript",
    "Svelte": "Svelte",
    "Vue": "Vue SFC (Single File Component)",
    "Vanilla JS": "JavaScript (no frameworks)",
}

state_managers = [
    "None",
    "Redux Toolkit",
    "XState",
    "Zustand",
    "MobX",
    "Pinia",
    "NgRx",
    "Jotai",
    "Svelte Stores",
]

print("Frameworks:", frameworks)
print("State managers:", [s for s in state_managers if s != "None"])

In [ ]:
for fw in frameworks:
    print(f"  {fw:12s} -> .{file_extensions[fw]:8s} ({code_languages[fw]})")

In [ ]:
OUTPUT_DIR = "converted_components"
os.makedirs(OUTPUT_DIR, exist_ok=True)


def save_output(code, framework, state_manager="None"):
    ext = file_extensions[framework]
    fw_slug = framework.lower().replace(" ", "-")
    if state_manager and state_manager != "None":
        sm_slug = state_manager.lower().replace(" ", "-")
        filename = f"TodoApp-{fw_slug}-{sm_slug}.{ext}"
    else:
        filename = f"TodoApp-{fw_slug}.{ext}"
    filepath = os.path.join(OUTPUT_DIR, filename)
    with open(filepath, "w") as f:
        f.write(code)
    return filepath


In [ ]:
# --- Step 1 prompts: React -> target framework ---

def system_prompt_for(framework):
    lang = code_languages[framework]
    return f"""You are an expert frontend developer. Your task is to convert React component code into equivalent {framework} component code.
Respond only with {lang} code in a single file. Do not provide any explanation other than occasional comments.
The converted component must reproduce identical functionality and appearance.
Use modern {framework} best practices and idiomatic patterns."""


def user_prompt_for(react_code, framework):
    lang = code_languages[framework]
    return f"""Convert this React component into a {framework} component with identical functionality and appearance.
Use modern {framework} best practices, idiomatic patterns, and proper typing where applicable.
Respond only with {lang} code.

React code to convert:

```jsx
{react_code}
```"""


# --- Step 2 prompts: add state manager to converted component ---

def state_mgr_system_prompt(framework, state_manager):
    return f"""You are an expert frontend developer specialising in {framework} and {state_manager}.
Your task is to refactor a {framework} component so that all of its state is managed by {state_manager}.
Respond only with complete, working code in a single file. Include any necessary imports.
Maintain identical functionality and appearance. Use modern {state_manager} best practices."""


def state_mgr_user_prompt(converted_code, framework, state_manager):
    return f"""Refactor this {framework} component to use {state_manager} for all state management.
Keep functionality and appearance identical. Use idiomatic {state_manager} patterns.
Respond only with code.

{framework} component to refactor:

```
{converted_code}
```"""

In [ ]:
def messages_for(react_code, framework):
    return [
        {"role": "system", "content": system_prompt_for(framework)},
        {"role": "user", "content": user_prompt_for(react_code, framework)},
    ]


def state_mgr_messages_for(converted_code, framework, state_manager):
    return [
        {"role": "system", "content": state_mgr_system_prompt(framework, state_manager)},
        {"role": "user", "content": state_mgr_user_prompt(converted_code, framework, state_manager)},
    ]
 

In [ ]:
def strip_code_fences(text):
    """Remove markdown code fences from LLM output."""
    import re
    text = re.sub(r'^```\w*\n?', '', text.strip())
    text = re.sub(r'\n?```$', '', text.strip())
    return text

In [ ]:
def convert(model, react_code, framework, state_manager):
    client = clients[model]
    reasoning_effort = "low" if 'gpt' in model else None

    # Step 1 — convert React to target framework
    response = client.chat.completions.create(
        model=model,
        messages=messages_for(react_code, framework),
        reasoning_effort=reasoning_effort,
    )
    code = strip_code_fences(response.choices[0].message.content)

    # Step 2 — integrate state manager (if one was selected)
    if state_manager and state_manager != "None":
        response = client.chat.completions.create(
            model=model,
            messages=state_mgr_messages_for(code, framework, state_manager),
            reasoning_effort=reasoning_effort,
        )
        code = strip_code_fences(response.choices[0].message.content)

    save_output(code, framework, state_manager)
    return code

In [ ]:
sample_react = """import React, { useState, useEffect } from 'react';

function TodoApp() {
  const [todos, setTodos] = useState([]);
  const [input, setInput] = useState('');
  const [filter, setFilter] = useState('all');

  useEffect(() => {
    const saved = localStorage.getItem('todos');
    if (saved) setTodos(JSON.parse(saved));
  }, []);

  useEffect(() => {
    localStorage.setItem('todos', JSON.stringify(todos));
  }, [todos]);

  const addTodo = () => {
    if (!input.trim()) return;
    setTodos([...todos, { id: Date.now(), text: input.trim(), done: false }]);
    setInput('');
  };

  const toggleTodo = (id) => {
    setTodos(todos.map(t => t.id === id ? { ...t, done: !t.done } : t));
  };

  const removeTodo = (id) => {
    setTodos(todos.filter(t => t.id !== id));
  };

  const filtered = todos.filter(t => {
    if (filter === 'active') return !t.done;
    if (filter === 'completed') return t.done;
    return true;
  });

  const remaining = todos.filter(t => !t.done).length;

  return (
    <div style={{ maxWidth: 480, margin: '2rem auto', fontFamily: 'sans-serif' }}>
      <h1>Todo List</h1>
      <div style={{ display: 'flex', gap: 8 }}>
        <input
          value={input}
          onChange={e => setInput(e.target.value)}
          onKeyDown={e => e.key === 'Enter' && addTodo()}
          placeholder="What needs to be done?"
          style={{ flex: 1, padding: 8 }}
        />
        <button onClick={addTodo}>Add</button>
      </div>
      <div style={{ margin: '1rem 0', display: 'flex', gap: 8 }}>
        {['all', 'active', 'completed'].map(f => (
          <button
            key={f}
            onClick={() => setFilter(f)}
            style={{ fontWeight: filter === f ? 'bold' : 'normal' }}
          >
            {f.charAt(0).toUpperCase() + f.slice(1)}
          </button>
        ))}
      </div>
      <ul style={{ listStyle: 'none', padding: 0 }}>
        {filtered.map(todo => (
          <li key={todo.id} style={{
            display: 'flex', alignItems: 'center',
            padding: '8px 0', borderBottom: '1px solid #eee'
          }}>
            <input
              type="checkbox"
              checked={todo.done}
              onChange={() => toggleTodo(todo.id)}
            />
            <span style={{
              flex: 1, marginLeft: 8,
              textDecoration: todo.done ? 'line-through' : 'none',
              color: todo.done ? '#999' : '#333'
            }}>
              {todo.text}
            </span>
            <button
              onClick={() => removeTodo(todo.id)}
              style={{ color: 'red', border: 'none', cursor: 'pointer' }}
            >
              x
            </button>
          </li>
        ))}
      </ul>
      {todos.length > 0 && (
        <p>{remaining} item{remaining !== 1 ? 's' : ''} remaining</p>
      )}
    </div>
  );
}

export default TodoApp;
"""

In [ ]:
from styles import CSS

with gr.Blocks(css=CSS, title="React Component Converter") as ui:
    gr.Markdown(
        "# React Component Converter\n"
        "Convert React components to **Angular**, **Svelte**, **Vue**, or **Vanilla JS** "
        "using Frontier LLMs — with optional **state manager** integration."
    )

    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            react_input = gr.Code(
                label="React (original)",
                value=sample_react,
                language="javascript",
                lines=30,
                elem_classes=["code-pane"],
            )
        with gr.Column(scale=6):
            framework_output = gr.Code(
                label="Converted Component",
                value="",
                language="javascript",
                lines=30,
                elem_classes=["code-pane"],
            )

    with gr.Row(elem_classes=["controls"]):
        model = gr.Dropdown(models, value=models[0], label="Model")
        framework = gr.Dropdown(frameworks, value=frameworks[0], label="Target Framework")
        state_mgr = gr.Dropdown(state_managers, value=state_managers[0], label="State Manager")
        convert_btn = gr.Button("Convert", elem_classes=["convert-btn"])

    convert_btn.click(
        fn=convert,
        inputs=[model, react_input, framework, state_mgr],
        outputs=[framework_output],
    )

ui.launch(inbrowser=True)
